In [25]:
! pip install bs4

In [1]:
import re
import pandas as pd
import numpy as np
import warnings 
import requests
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

# Flipkart Laptop price analysis.

# 1.Search for relavant website

In [2]:
url = 'https://www.flipkart.com/search?q=laptop'

In [3]:
url = 'https://www.flipkart.com/search?q=laptop'
headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    }
# Loading  
response = requests.get(url, headers=headers)
# response = request.get(url)
response

<Response [200]>

In [4]:
response.status_code

200

In [5]:
html_code=response.text

In [6]:
html_code[:2000]

'<!doctype html><html lang="en"><head><link href="https://rukminim2.flixcart.com" rel="preconnect"/><link rel="stylesheet" href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/bundle.aa1465.css"/><link rel="stylesheet" href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/bundle.fbaef9.css"/><meta http-equiv="Content-type" content="text/html; charset=utf-8"/><meta http-equiv="X-UA-Compatible" content="IE=Edge"/><meta property="fb:page_id" content="102988293558"/><meta property="fb:admins" content="658873552,624500995,100000233612389"/><link rel="shortcut icon" href="https://static-assets-web.flixcart.com/www/promos/new/20150528-140547-favicon-retina.ico"/><link type="application/opensearchdescription+xml" rel="search" href="/osdd.xml?v=2"/><meta property="og:type" content="website"/><meta name="og_site_name" property="og:site_name" content="Flipkart.com"/><link rel="apple-touch-icon" sizes="57x57" href="/apple-touch-icon-57x57.png"/><link rel="apple

In [7]:
soup= BeautifulSoup(html_code)

 # 2.Extract data

In [8]:
# Product title

title = soup.find("div", attrs = {'class':'RG5Slk'})
title.text

'HP MSO 2024 Intel Core i3 13th Gen 1315U - (16 GB/512 GB SSD/Windows 11 Home) 15-fd0574TU / 15 - fd066...'

In [9]:
# Product rating

rating= soup.find("div", attrs = {'class':'MKiFS6'})
rating.text

'4.2'

In [10]:
# Product Price
price = soup.find("div", attrs = {'class':'hZ3P6w DeU9vF'})
price.text

'₹40,990'

In [11]:
# product reviews
reviews = soup.find('span',attrs = {'class':'PvbNMB'})
reviews.text

'2,454 Ratings\xa0&\xa0147 Reviews'

In [12]:
# product feature list

features_list = soup.find('ul',attrs = {'class':'HwRTzP'})
features_list.text

'Intel Core i3 Processor (13th Gen)16 GB DDR4 RAMWindows 11 Home Operating System512 GB SSD39.62 cm (15.6 Inch) DisplayMS Office Home 2024 + MISC PC Game Pass DA 3M1 Year Onsite Warranty'

In [13]:
discount=soup.find("div",attrs={"class":"HQe8jr"})
discount.text

'19% off'

In [15]:
title_data = []
price_data = []
rating_data = []
review_data = []
feature_data = []
discount_data=[]


for i in range(1,42):
    #step 1 : Identify the url
    url = f'https://www.flipkart.com/search?q=laptops&page={i}'
    # step 2 : Extract the HTML code
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers = headers)
    if response.status_code == 403:
        print("There's a 403 Error!")
        break

# Step 3 : Parse the HTML code to Beautifulsoup object
    html_code = response.text
    soup = BeautifulSoup(html_code)
    containers = soup.find_all('div',attrs = {'class':'ZFwe0M row'})
    for c in containers:
        # for each container scrape all data
        # Product title
        title = c.find('div', attrs = {'class':'RG5Slk'})
        if title is None:
            title_data.append(np.nan)
        else:
            title_data.append(title.text)

# Product Price
        price = c.find('div', attrs = {'class': 'hZ3P6w DeU9vF'}) 
        if price is None: 
            price_data.append(np.nan) 
        else: 
            price_data.append(price.text)

# Customer rating and review
        review = c.find( "span" ,attrs = {"class" : 'PvbNMB'}) 
        if review is None: 
            review_data.append(np.nan) 
        else: 
            review_data.append(review.text)

# rating
        rating = c.find('div', attrs = {'class': 'MKiFS6'})
        if rating is None:
            rating_data.append(np.nan)
        else:
            rating_data.append(rating.text)

# features
        features = c.find("ul",attrs={"class":"HwRTzP"})
        if features is None:
            feature_data.append(np.nan)
        else:
            feature_data.append(features.text)

        discount=c.find("div",attrs={"class":"HQe8jr"})
        if discount is None:
            discount_data.append(np.nan)
        else:
            discount_data.append(discount.text)

In [ ]:
print("Product Title",len(title_data)) 
print("Product Price",len(price_data))
print("Product Rating",len(rating_data)) 
print("Product Review",len(review_data)) 
print("Product Feature",len(feature_data))
print("Product Discount",len(discount_data))

# 3.Create a Data Frame

In [16]:
import pandas as pd  
df = pd. DataFrame({'Product Title':title_data, 
                    'Product Price': price_data, 
                    'Product Rating': rating_data, 
                    'Product Review': review_data, 
                    'Product Feature': feature_data,
                   'Product Discount':discount_data}) 
                    
df.head()  

,Product Title,Product Price,Product Rating,Product Review,Product Feature,Product Discount
0,HP MSO 2024 Intel Core i3 13th Gen 1315U - (16...,"₹40,990",4.2,"2,454 Ratings & 147 Reviews",Intel Core i3 Processor (13th Gen)16 GB DDR4 R...,19% off
1,ASUS Expertbook P1 High-performance processor ...,"₹55,990",4.3,"4,183 Ratings & 324 Reviews",Intel Core i5 Processor (13th Gen)32 GB DDR5 R...,51% off
2,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,"₹24,699",3.8,"7,917 Ratings & 693 Reviews",Intel Celeron Dual Core Processor8 GB DDR4 RAM...,31% off
3,"HP 15 Laptop with Backlit Keyboard & MSO'2024,...","₹55,990",4.5,28 Ratings & 5 Reviews,AMD Ryzen 7 Octa Core Processor16 GB DDR4 RAMW...,11% off
4,Acer Aspire 3 Backlit AMD Ryzen 7 Octa Core 77...,"₹43,990",4.1,"1,638 Ratings & 123 Reviews",AMD Ryzen 7 Octa Core Processor16 GB DDR4 RAMW...,45% off


In [17]:
df.shape

(888, 6)

In [18]:
df.shape

(888, 6)

In [ ]:
df.columns

# 4.Export into .csv format

In [19]:
df.to_csv('laptop_raw_data.csv',index=False)